In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import optuna


/Users/apple/ai-bootcamp-2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Load Dataser
data = load_breast_cancer()
X, y = data.data, data.target

#Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Standardize Features
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

print(f"Train data shpae: {X_train.shape}")
print(f"Test data shpae: {X_test.shape}")

Train data shpae: (455, 30)
Test data shpae: (114, 30)


In [5]:
#Train Baseline XGBoost model
baseline_model = XGBClassifier(eval_metric ='logloss', random_state = 42)
baseline_model.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [6]:
#Evaluate Baseline Model
baseline_pred = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print(f"Baseline Model Accuracy: {baseline_accuracy:.4f}")

Baseline Model Accuracy: 0.9561


In [24]:
#Define Objective for optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 3, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0, log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0, log=False),
        'gamma': trial.suggest_float('gamma', 0.001, 5, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10, log=True),
    }
    #Train XGBoost model with suggested params
    model = XGBClassifier(eval_metric='logloss', random_state=42, **params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy
    
#Create an Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

#Get best parameters
best_params = study.best_params
best_value = study.best_value
print(f"Best Parameters: {best_params}")
print(f"Best Accuracy: {best_value:.4f}")

[I 2026-01-09 12:50:51,222] A new study created in memory with name: no-name-2718ff30-56cf-403f-8383-eb668c84a03b
[I 2026-01-09 12:50:51,248] Trial 0 finished with value: 0.9649122807017544 and parameters: {'n_estimators': 42, 'max_depth': 10, 'learning_rate': 0.15063047389812995, 'subsample': 0.5240924567927069, 'colsample_bytree': 0.9231843976491241, 'gamma': 0.4290850085337377, 'reg_alpha': 0.30595995726522424, 'reg_lambda': 0.007271851183738689}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-01-09 12:50:51,273] Trial 1 finished with value: 0.9385964912280702 and parameters: {'n_estimators': 51, 'max_depth': 9, 'learning_rate': 0.035216258488208724, 'subsample': 0.8349060505109573, 'colsample_bytree': 0.7193418682191941, 'gamma': 0.12181091740505628, 'reg_alpha': 9.149480025657938, 'reg_lambda': 4.583613740086209}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-01-09 12:50:51,294] Trial 2 finished with value: 0.9649122807017544 and parameters: {'n_estimators': 72,

Best Parameters: {'n_estimators': 73, 'max_depth': 10, 'learning_rate': 0.12774158070036123, 'subsample': 0.601096370033936, 'colsample_bytree': 0.8489511834789645, 'gamma': 0.010894281840342935, 'reg_alpha': 0.2170053308259569, 'reg_lambda': 5.610032798217878}
Best Accuracy: 0.9737


In [25]:
#Define Parameters Grid
param_grid = {
    'n_estimators': [3, 10, 20, 50, 100],
    'max_depth': [3, 5, 10, 20],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.5, 0.7, 0.9, 1.0]
}

In [26]:
#Train XGBoost with Search Grid
grid_search = GridSearchCV(
    estimator = XGBClassifier(eval_metric = 'logloss', random_state = 42),
    param_grid = param_grid,
    cv = 3,
    scoring = 'accuracy',
    verbose = 1
)
grid_search.fit(X_train, y_train)

#Get Best Parameters
best_params = grid_search.best_params_
best_accuracy = grid_search.best_score_
print(f"\n\nGrid Search Best Parameters: {best_params}")
print(f"Grid Search Best Accuracy: {best_accuracy:.4f}")

Fitting 3 folds for each of 320 candidates, totalling 960 fits


Grid Search Best Parameters: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.7}
Grid Search Best Accuracy: 0.9758


In [27]:
#Define Parameters Distribution
param_dist = {
    'n_estimators': [50,100,200,300,400],
    'max_depth': [3, 5, 7, 9, 11],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator = XGBClassifier(eval_metric = 'logloss', random_state = 42),
    param_distributions = param_dist,
    n_iter = 50,
    cv = 3,
    scoring = 'accuracy',
    verbose = 1,
    random_state = 42
)
random_search.fit(X_train, y_train)

#Get Best Parameters
best_params = random_search.best_params_
best_accuracy = random_search.best_score_
print(f"\n\nRandom Search Best Parameters: {best_params}")
print(f"Random Search Best Accuracy: {best_accuracy:.4f}")

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2229783402.py, line 7)